In [ ]:
import torch
import torch.nn as nn
import numpy as np

class PyTorchDynamicNetwork(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, max_neurons: int = 2000, steps: int = 3):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.max_neurons = max_neurons
        self.steps = steps

        # Generation 2: Output indices track which neurons are output heads
        self.output_indices = list(range(input_dim, input_dim + output_dim))
        
        # Start with input + output + 4 hidden neurons
        self.active_neurons = input_dim + output_dim + 4
        self.free_neurons = set()

        # Adjacency Matrix (Structure)
        self.register_buffer('M', torch.zeros(max_neurons, max_neurons))
        # Trainable Mask (Memory Protection)
        self.register_buffer('trainable_mask', torch.ones(max_neurons, max_neurons))
        
        # Generation 2: Localized Per-Neuron Stress
        self.register_buffer('neuron_stress', torch.zeros(max_neurons))
        
        # Anomaly Detection: Loss Exponential Moving Averages
        self.loss_ema = 0.0
        self.fast_loss_ema = 0.0
        self.slow_loss_ema = 0.0

        # Weight Matrix and Bias
        self.W = nn.Parameter(torch.zeros(max_neurons, max_neurons))
        self.b = nn.Parameter(torch.zeros(max_neurons))

        self._hook_registered = False
        self._init_random_connections()

    def _init_random_connections(self):
        with torch.no_grad():
            for i in range(self.active_neurons):
                for j in range(self.active_neurons):
                    # Neurons can't connect to themselves, and inputs receive no connections
                    if i != j and i >= self.input_dim:
                        if torch.rand(1).item() > 0.5:
                            self.M[i, j] = 1.0
            
            # Initialize weights where connections exist
            mask = self.M[:self.active_neurons, :self.active_neurons].bool()
            self.W[:self.active_neurons, :self.active_neurons][mask] = torch.randn(mask.sum()) * 0.1

    def _register_hooks(self):
        def _w_hook(grad):
            return grad * self.trainable_mask * self.M

        def _b_hook(grad):
            # If a neuron's incoming connections are ALL frozen, freeze its bias too
            b_mask = torch.ones_like(grad)
            for i in range(self.active_neurons):
                # If it has incoming connections but ALL of them are frozen (trainable=0)
                if self.M[i, :].sum() > 0 and (self.M[i, :] * self.trainable_mask[i, :]).sum() == 0:
                    b_mask[i] = 0.0
            return grad * b_mask

        self.W.register_hook(_w_hook)
        self.b.register_hook(_b_hook)
        self._hook_registered = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self._hook_registered:
            self._register_hooks()

        batch = x.size(0)
        state = torch.zeros(batch, self.max_neurons, device=x.device)
        state[:, :self.input_dim] = x

        W_eff = self.W * self.M
        for _ in range(self.steps):
            new_state = torch.relu(torch.matmul(state, W_eff.T) + self.b)
            # Only update non-input neurons
            state = torch.cat([x, new_state[:, self.input_dim:]], dim=1)

        # Generation 2: Return output based on flexible indices
        return state[:, self.output_indices]

    def update_stress(self, grads_task: torch.Tensor, grads_replay: torch.Tensor, beta: float = 0.9):
        """Generation 2: Computes stress PER NEURON based on incoming gradient conflict."""
        with torch.no_grad():
            # Measure how much the new gradients destroy the old knowledge (negative dot product)
            conflict = -(grads_task * grads_replay)
            conflict = torch.relu(conflict)
            
            # Sum over incoming connections (dim=1 in target-source matrix)
            neuron_conflict = conflict.sum(dim=1)
            
            # EMA Update
            self.neuron_stress = beta * self.neuron_stress + (1 - beta) * neuron_conflict

    def get_max_neuron_stress(self) -> float:
        """Returns the highest stress level among all active neurons."""
        if self.active_neurons == 0: return 0.0
        return self.neuron_stress[:self.active_neurons].max().item()

    def update_loss_ema(self, current_loss: float, beta: float = 0.9, fast_beta: float = 0.5, slow_beta: float = 0.99):
        """Updates the running loss EMAs for anomaly detection."""
        self.loss_ema = beta * self.loss_ema + (1 - beta) * current_loss
        self.fast_loss_ema = fast_beta * self.fast_loss_ema + (1 - fast_beta) * current_loss
        self.slow_loss_ema = slow_beta * self.slow_loss_ema + (1 - slow_beta) * current_loss

    def detect_task_shift(self, max_stress: float, margin: float = 0.2, stress_threshold: float = 0.2) -> bool:
        """
        Anomaly Detection:
        If the Fast EMA of loss diverges from the Slow EMA (MACD), OR if Gradient 
        Conflict (max_stress) exceeds the threshold, a distribution shift is detected.
        """
        # Wait until the slow EMA has built up some history (Burn-In)
        if self.slow_loss_ema < 0.01: 
            return False
            
        macd_spike = (self.fast_loss_ema - self.slow_loss_ema) > margin
        stress_spike = max_stress > stress_threshold
        
        return macd_spike or stress_spike

    def stress_freeze(self, base_threshold: float = 0.5, sensitivity_factor: float = 0.4) -> tuple:
        """
        Localized Parameter Freezing:
        Freezes incoming weights for neurons experiencing high gradient conflict.
        The threshold dynamically lowers as the running loss increases (higher sensitivity).
        Returns: (n_frozen, dynamic_threshold)
        """
        n_frozen = 0
        
        # Calculate dynamic threshold: Higher loss = Lower threshold (higher sensitivity)
        capped_loss = min(self.loss_ema, 1.0)
        dynamic_threshold = max(0.05, base_threshold - (sensitivity_factor * capped_loss))
        
        with torch.no_grad():
            stressed_neurons = (self.neuron_stress > dynamic_threshold) & (torch.arange(self.max_neurons, device=self.neuron_stress.device) < self.active_neurons)
            
            for i in torch.where(stressed_neurons)[0]:
                i = i.item()
                # Find its incoming connections that are currently trainable
                incoming = (self.trainable_mask[i, :] == 1) & (self.M[i, :] == 1)
                num_to_freeze = incoming.sum().item()
                
                if num_to_freeze > 0:
                    self.trainable_mask[i, incoming] = 0.0
                    n_frozen += num_to_freeze
                    # Reset stress since it is now protected
                    self.neuron_stress[i] = 0.0
                    
        return n_frozen, dynamic_threshold

    def grow_neuron(self, num_connections: int = 5):
        if len(self.free_neurons) > 0:
            new_idx = self.free_neurons.pop()
        else:
            if self.active_neurons >= self.max_neurons:
                return -1
            new_idx = self.active_neurons
            self.active_neurons += 1
            
        with torch.no_grad():
            self.M[new_idx, :] = 0.0
            self.M[:, new_idx] = 0.0
            self.trainable_mask[new_idx, :] = 1.0
            self.trainable_mask[:, new_idx] = 1.0
            self.neuron_stress[new_idx] = 0.0
            
            # Incoming connections from valid sources
            valid_sources = list(range(self.input_dim)) + [
                n for n in range(self.input_dim + self.output_dim, self.max_neurons) 
                if n < self.active_neurons and n not in self.free_neurons and n != new_idx
            ]
            if len(valid_sources) > 0:
                n_conn = min(num_connections, len(valid_sources))
                sources = torch.randperm(len(valid_sources))[:n_conn]
                for s in sources:
                    self.M[new_idx, valid_sources[s.item()]] = 1.0
            
            # Outgoing connections to valid targets
            valid_targets = [
                n for n in range(self.input_dim, self.max_neurons)
                if n < self.active_neurons and n not in self.free_neurons and n != new_idx
            ]
            if len(valid_targets) > 0:
                n_conn = min(num_connections, len(valid_targets))
                targets = torch.randperm(len(valid_targets))[:n_conn]
                for t in targets:
                    self.M[valid_targets[t.item()], new_idx] = 1.0

            # Initialize weights
            in_mask = self.M[new_idx, :].bool()
            self.W[new_idx, in_mask] = torch.randn(in_mask.sum()) * 0.1
            out_mask = self.M[:, new_idx].bool()
            self.W[out_mask, new_idx] = torch.randn(out_mask.sum()) * 0.1
            self.b[new_idx] = 0.0
            
        return new_idx

    def prune_dead_neurons(self, threshold: float = 1e-3) -> int:
        n_pruned = 0
        with torch.no_grad():
            effective_W = self.W * self.M
            for idx in range(self.input_dim + self.output_dim, self.active_neurons):
                if idx in self.free_neurons:
                    continue
                # Calculate L1 norm of incoming weights
                incoming_l1 = torch.sum(torch.abs(effective_W[idx, :])).item()
                if incoming_l1 < threshold:
                    # Turn off the neuron
                    self.M[idx, :] = 0.0
                    self.M[:, idx] = 0.0
                    self.free_neurons.add(idx)
                    n_pruned += 1
        return n_pruned

    def merge_redundant_neurons(self, similarity_threshold: float = 0.95) -> int:
        n_merged = 0
        with torch.no_grad():
            effective_W = self.W * self.M
            active_hiddens = [
                idx for idx in range(self.input_dim + self.output_dim, self.active_neurons)
                if idx not in self.free_neurons
            ]
            
            for i in range(len(active_hiddens)):
                idx_i = active_hiddens[i]
                if idx_i in self.free_neurons:
                    continue
                
                vec_i = effective_W[idx_i, :]
                norm_i = torch.norm(vec_i)
                if norm_i == 0:
                    continue
                    
                for j in range(i + 1, len(active_hiddens)):
                    idx_j = active_hiddens[j]
                    if idx_j in self.free_neurons:
                        continue
                        
                    vec_j = effective_W[idx_j, :]
                    norm_j = torch.norm(vec_j)
                    if norm_j == 0:
                        continue
                        
                    sim = torch.dot(vec_i, vec_j) / (norm_i * norm_j)
                    if sim > similarity_threshold:
                        # Combine output weights of j into i
                        self.W[:, idx_i] += self.W[:, idx_j]
                        # Turn off neuron j
                        self.M[idx_j, :] = 0.0
                        self.M[:, idx_j] = 0.0
                        self.free_neurons.add(idx_j)
                        n_merged += 1
                        
        return n_merged

    def compress_network(self, prune_threshold: float = 1e-3, similarity_threshold: float = 0.95) -> tuple:
        n_pruned = self.prune_dead_neurons(prune_threshold)
        n_merged = self.merge_redundant_neurons(similarity_threshold)
        return n_pruned, n_merged

    def grow_output_head(self, num_connections: int = 15):
        """Generation 2: Dynamically spawns a brand new Output Neuron (Multi-Head)."""
        if self.active_neurons >= self.max_neurons:
            return -1
            
        new_idx = self.active_neurons
        self.active_neurons += 1
        self.output_dim += 1
        self.output_indices.append(new_idx)
        
        with torch.no_grad():
            self.W[new_idx, :] = 0.0
            
            # Incoming connections: from hidden neurons only (or inputs)
            hidden_and_input = list(range(self.input_dim)) + [i for i in range(new_idx) if i not in self.output_indices[:-1]]
            
            if len(hidden_and_input) > 0:
                k = min(num_connections, len(hidden_and_input))
                chosen = torch.tensor(hidden_and_input)[torch.randperm(len(hidden_and_input))[:k]]
                self.M[new_idx, chosen] = 1.0
                self.W[new_idx, chosen] = torch.randn(k, device=self.W.device) * 0.1
                
        return new_idx

    def n_trainable(self) -> int:
        return int((self.M[:self.active_neurons, :self.active_neurons] * self.trainable_mask[:self.active_neurons, :self.active_neurons]).sum().item())
        
    def n_frozen(self) -> int:
        return int((self.M[:self.active_neurons, :self.active_neurons] * (1 - self.trainable_mask[:self.active_neurons, :self.active_neurons])).sum().item())



In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json


class ReplayBuffer:
    def __init__(self, capacity: int = 500):
        self.capacity = capacity
        self.X = None
        self.y = None
        self.task_ids = None

    def add_data(self, X_new: torch.Tensor, y_new: torch.Tensor, task_id: int, num_samples: int = 16):
        idx = torch.randperm(X_new.size(0))[:num_samples]
        X_sub = X_new[idx].clone().detach()
        y_sub = y_new[idx].clone().detach()
        t_sub = torch.full((num_samples,), task_id, dtype=torch.long)

        if self.X is None:
            self.X = X_sub
            self.y = y_sub
            self.task_ids = t_sub
        else:
            self.X = torch.cat([self.X, X_sub], dim=0)
            self.y = torch.cat([self.y, y_sub], dim=0)
            self.task_ids = torch.cat([self.task_ids, t_sub], dim=0)

            if self.X.size(0) > self.capacity:
                keep = torch.randperm(self.X.size(0))[:self.capacity]
                self.X = self.X[keep]
                self.y = self.y[keep]
                self.task_ids = self.task_ids[keep]

    def sample(self, batch_size: int = 64) -> tuple:
        if self.X is None:
            return None, None, None
        size = self.X.size(0)
        idx = torch.randint(0, size, (min(batch_size, size),))
        return self.X[idx], self.y[idx], self.task_ids[idx]

    def has_data(self) -> bool:
        return self.X is not None and self.X.size(0) > 0

def run_experiment(seed, config_name, use_replay, use_growth, use_freeze, use_compress):
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    device = torch.device('cpu')

    # 1. Generate Data Stream
    def make_circles(n_samples, noise=0.05, factor=0.5):
        theta = np.random.uniform(0, 2*np.pi, n_samples)
        r = np.where(np.random.rand(n_samples) > 0.5, 1.0, factor)
        x = r * np.cos(theta) + np.random.randn(n_samples) * noise
        y = r * np.sin(theta) + np.random.randn(n_samples) * noise
        labels = (r == 1.0).astype(int)
        return torch.FloatTensor(np.c_[x, y]), torch.FloatTensor(labels).unsqueeze(1)

    def make_linear(n_samples, noise=0.05):
        x = np.random.uniform(-1.5, 1.5, n_samples)
        y = np.random.uniform(-1.5, 1.5, n_samples)
        labels = (x + y > 0).astype(int)
        X = np.c_[x + np.random.randn(n_samples)*noise, y + np.random.randn(n_samples)*noise]
        return torch.FloatTensor(X), torch.FloatTensor(labels).unsqueeze(1)

    def make_xor(n_samples, noise=0.05):
        x = np.random.uniform(-1.5, 1.5, n_samples)
        y = np.random.uniform(-1.5, 1.5, n_samples)
        labels = ((x > 0) ^ (y > 0)).astype(int)
        X = np.c_[x + np.random.randn(n_samples)*noise, y + np.random.randn(n_samples)*noise]
        return torch.FloatTensor(X), torch.FloatTensor(labels).unsqueeze(1)

    tasks = [
        ("Circles", *make_circles(4000)),
        ("Linear", *make_linear(4000)),
        ("XOR", *make_xor(4000))
    ]

    stream_X = torch.cat([t[1] for t in tasks])
    stream_y = torch.cat([t[2] for t in tasks])
    
    # Track true task boundaries (batches)
    batch_size = 16
    task_boundaries = [
        0, 
        4000 // batch_size, 
        8000 // batch_size
    ]
    
    dynamic = PyTorchDynamicNetwork(input_dim=2, output_dim=1, max_neurons=500).to(device)
    
    # If not using growth, we start with a larger static capacity to be fair
    if not use_growth:
        for _ in range(50):
            dynamic.grow_neuron(10)
            
    initial_neurons = dynamic.active_neurons
    peak_active = initial_neurons
    total_created = 0
    total_pruned = 0
    
    # Add weight decay so useless weights naturally decay towards zero, allowing them to be pruned!
    optimizer = torch.optim.Adam(dynamic.parameters(), lr=0.01, weight_decay=1e-4)
    criterion = nn.MSELoss()
    replay = ReplayBuffer(capacity=500)
    
    stream_loss_ema = 0.5
    batches_since_grow = 0
    total_batches = 0
    
    # Metrics
    eval_matrix = [] # [batch_idx, task0_acc, task1_acc, task2_acc]
    detection_delays = []
    current_true_task = 0
    
    def evaluate():
        accs = []
        for t_id, (name, X, y) in enumerate(tasks):
            # Strict Single Head Evaluation (No Task ID Leakage)
            acc = ((dynamic(X)[:, 0].unsqueeze(1) > 0.5).float() == y).float().mean().item()
            accs.append(acc)
        return accs

    for i in range(0, stream_X.size(0), batch_size):
        if total_batches in task_boundaries:
            if total_batches > 0:
                current_true_task += 1
                
        bx = stream_X[i:i+batch_size].to(device)
        by = stream_y[i:i+batch_size].to(device)
        
        current_active = dynamic.active_neurons - len(getattr(dynamic, 'free_neurons', []))
        if current_active > peak_active:
            peak_active = current_active
            
        optimizer.zero_grad()
        
        pred_all_heads = dynamic(bx)
        # All tasks forced through the same shared bottleneck
        pred = pred_all_heads[:, 0].unsqueeze(1)
        loss = criterion(pred, by)
        
        if use_replay and replay.has_data():
            rx, ry, _ = replay.sample(64) # We don't care about task IDs from replay anymore
            rx, ry = rx.to(device), ry.to(device)
            r_pred_all = dynamic(rx)
            
            # Replay also forced through shared head
            r_pred = r_pred_all[:, 0].unsqueeze(1)
                
            loss_r = criterion(r_pred, ry)
            
            g_t = torch.autograd.grad(loss, dynamic.W, retain_graph=True, allow_unused=True)[0]
            g_r = torch.autograd.grad(loss_r, dynamic.W, retain_graph=True, allow_unused=True)[0]
            
            if g_t is not None and g_r is not None:
                dynamic.update_stress(g_t, g_r)
                
            loss = loss + loss_r
            
        loss.backward()
        optimizer.step()
        
        dynamic.update_loss_ema(loss.item())
        stream_loss_ema = 0.9 * stream_loss_ema + 0.1 * loss.item()
        
        total_batches += 1
        batches_since_grow += 1
        max_stress = dynamic.get_max_neuron_stress()
        
        # Autonomous Shift Detection
        if use_growth:
            shift_detected = dynamic.detect_task_shift(max_stress, margin=0.15, stress_threshold=0.2)
            
            if shift_detected and total_batches > 50 and batches_since_grow > 50:
                # Record delay
                if current_true_task > 0 and current_true_task < len(task_boundaries):
                    delay = total_batches - task_boundaries[current_true_task]
                    if delay > 0 and (len(detection_delays) < current_true_task):
                        detection_delays.append(delay)
                
                dynamic.fast_loss_ema = stream_loss_ema
                dynamic.slow_loss_ema = stream_loss_ema
                batches_since_grow = 0
                
        # Freezing
        if use_freeze:
            capped_loss = min(dynamic.loss_ema, 1.0)
            current_threshold = max(0.05, 0.5 - (1.0 * capped_loss))
            dynamic.current_threshold = current_threshold
            
            if max_stress > current_threshold:
                n_frozen, _ = dynamic.stress_freeze(base_threshold=0.5, sensitivity_factor=1.0)
                if n_frozen > 0:
                    ret = dynamic.grow_neuron(10)
                    if ret != -1:
                        total_created += 1
                    batches_since_grow = 0
                    
        # Capacity Expansion
        if use_growth and stream_loss_ema > 0.2 and batches_since_grow > 20 and max_stress <= getattr(dynamic, 'current_threshold', 0.5):
            ret = dynamic.grow_neuron(10)
            if ret != -1:
                total_created += 1
            batches_since_grow = 0
                
        if use_replay and torch.rand(1).item() < 0.10:
            replay.add_data(bx, by, 0, num_samples=16)
            
        # Periodic Evaluation (every 20 batches)
        if total_batches % 20 == 0:
            eval_matrix.append(evaluate())
            
        # Periodic Compression
        if use_compress and total_batches % 100 == 0:
            p, m = dynamic.compress_network(prune_threshold=0.15, similarity_threshold=0.50)
            total_pruned += (p + m)
            
    # Final Evaluation
    final_accs = evaluate()
    eval_matrix = np.array(eval_matrix)
    
    # Calculate Backward Transfer
    # For each task, BT = final_acc - max_acc_during_its_training_window
    bt = 0.0
    for t_id in range(len(tasks)):
        # Max acc achieved during the entire run
        max_acc = np.max(eval_matrix[:, t_id])
        bt += (final_accs[t_id] - max_acc)
    bt /= len(tasks)
    
    metrics = {
        'final_accuracy': np.mean(final_accs),
        'backward_transfer': bt,
        'net_growth': (dynamic.active_neurons - len(getattr(dynamic, 'free_neurons', []))) - initial_neurons,
        'peak_growth': peak_active - initial_neurons,
        'total_created': total_created,
        'total_pruned': total_pruned,
        'detection_delay_avg': np.mean(detection_delays) if detection_delays else -1
    }
    
    return metrics

if __name__ == "__main__":
    configs = [
        ("Static MLP (Naive)", False, False, False, False),
        ("Static MLP (Replay)", True, False, False, False),
        ("Dynamic (Growth Only) + Replay", True, True, False, False),
        ("Dynamic (Freeze Only) + Replay", True, False, True, False),
        ("Full Dynamic (Growth+Freeze) + Replay", True, True, True, False),
        ("Full Dynamic + Compress", True, True, True, True)
    ]
    
    results = {}
    
    for name, ur, ug, uf, uc in configs:
        print(f"\\nRunning: {name}")
        metrics_acc = []
        metrics_bt = []
        metrics_net = []
        metrics_peak = []
        metrics_created = []
        metrics_pruned = []
        metrics_delay = []
        
        for seed in [42, 100, 999]:
            m = run_experiment(seed, name, ur, ug, uf, uc)
            metrics_acc.append(m['final_accuracy'])
            metrics_bt.append(m['backward_transfer'])
            metrics_net.append(m['net_growth'])
            metrics_peak.append(m['peak_growth'])
            metrics_created.append(m['total_created'])
            metrics_pruned.append(m['total_pruned'])
            if m['detection_delay_avg'] != -1:
                metrics_delay.append(m['detection_delay_avg'])
                
        results[name] = {
            'Accuracy': f"{np.mean(metrics_acc)*100:.1f}% ± {np.std(metrics_acc)*100:.1f}%",
            'Forgetting': f"{np.mean(metrics_bt)*100:.1f}% ± {np.std(metrics_bt)*100:.1f}%",
            'Net Growth': f"{np.mean(metrics_net):.1f}",
            'Peak Growth': f"{np.mean(metrics_peak):.1f}",
            'Created': f"{np.mean(metrics_created):.1f}",
            'Pruned': f"{np.mean(metrics_pruned):.1f}",
            'Delay': f"{np.mean(metrics_delay):.1f}" if len(metrics_delay) > 0 else "N/A"
        }
    
    print("\\n\\n### Comprehensive Benchmark Results\\n")
    print("| Configuration | Final Accuracy | Forgetting (BT) | Net Growth | Peak Growth | Created | Pruned | Shift Detection Delay |")
    print("|--------------|----------------|----------------|------------|-------------|---------|--------|-----------------------|")
    for name, r in results.items():
        print(f"| {name} | {r['Accuracy']} | {r['Forgetting']} | {r['Net Growth']} | {r['Peak Growth']} | {r['Created']} | {r['Pruned']} | {r['Delay']} |")

